In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from limb_fitting import *
from ellipse import *

In [7]:
def SM(r, a, center=(0,0)):
    xc, yc = center
    R = np.array([[1 + np.cos(2 * a), np.sin(2 * a), -((1 + np.cos(2 * a)) * xc + np.sin(2 * a) * yc)],
                  [np.sin(2 * a), 1 - np.cos(2 * a), -(np.sin(2 * a) * xc + (1 - np.cos(2 * a)) * yc)],
                  [0, 0, 0]])
    return np.identity(3) + (r - 1) / 2 * R

def elli_plot(ellis, *, ext=1, angle=0):
    from matplotlib import patches
    M = SM(ext, angle)

    fig, ax = plt.subplots(figsize=(8,8))

    ellis_ = []

    for elli in ellis:
        elli = Ellipse.from_matrix(M.T @ elli.matrix @ M)

        ellis_ += [elli]

        A, B, C = elli[:3]

        lam1 = ((C + A) + np.sqrt((A - C) ** 2 + B ** 2)) / 2
        lam2 = ((C + A) - np.sqrt((A - C) ** 2 + B ** 2)) / 2

        #print((lam1 - lam2) / (lam1 + lam2))

        v1 = np.array([A - lam2, B / 2])
        v2 = np.array([A - lam1, B / 2])
        v1 /= np.linalg.norm(v1)
        v2 /= np.linalg.norm(v2)


        e = elli.flatness
        #print(e)
        k = 1 + 500 * e
        a = 60 * np.sqrt(k)
        b = 60 / np.sqrt(k)
        b_ = 1000

        xc, yc = elli.center
        #a, b = elli.major, elli.minor


        ax.plot([xc - v1[0] * b_ / 2, xc + v1[0] * b_ / 2],
                [yc - v1[1] * b_ / 2, yc + v1[1] * b_ / 2], color='black', lw=0.2)

        ax.plot([xc - v2[0] * a / 2, xc + v2[0] * a / 2],
                [yc - v2[1] * a / 2, yc + v2[1] * a / 2], color='blue', ls='--', lw=0.7)

        ax.plot([xc - v1[0] * b / 2, xc + v1[0] * b / 2],
                [yc - v1[1] * b / 2, yc + v1[1] * b / 2], color='blue', ls='--', lw=0.7)

        ellipse = patches.Ellipse((xc, yc), a, b, angle=elli.theta * 180 / np.pi, fill=False, edgecolor='red', lw=1)
        ax.add_patch(ellipse)

    plt.xlim(700, 1400)
    plt.ylim(700, 1400)
    plt.show()
    plt.tight_layout()

    #return ellis_

In [8]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T000503_V202512011636C_0569230100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001105_V202511212307C_0569230125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001705_V202511212307C_0569230150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002305_V202511212307C_0569230175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002905_V202511212307C_0569230200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T003505_V202511212307C_0569230225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T004105_V202511212338C_0569230250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [4]:
ellis = []

for file in files:
    with fits.open(file) as hdul:
        image = hdul[0].data[0]

    edges = find_edges(image)
    x, y = np.where(edges)
    x, y = filter_outliers(x, y, acc=2)
    elli = fit_ellipse(x, y)
    ellis += [elli]

In [9]:
elli_plot(ellis, ext=1.001, angle=-0.567)